In [ ]:
suppressPackageStartupMessages({
  library(SpatialFeatureExperiment)
  library(SFEData)
  library(scater)
})

In [ ]:
ExperimentHub::setExperimentHubOption('cache', './experiment_hub/')
# Load the dataset
sfe <- SFEData::McKellarMuscleData(dataset = "small", file_path = "./")
# Take spots that are covered with tissue
sfe_tissue <- sfe[, colData(sfe)$in_tissue]
# Filter out genes with no counts
sfe_tissue <- sfe_tissue[rowSums(counts(sfe_tissue)) > 0, ]
# Convert counts log-transformed normalized expression values
sfe_tissue <- scater::logNormCounts(sfe_tissue)

In [ ]:

# Convert to AnnData object
# ann <-  zellkonverter::SCE2AnnData(sfe_tissue)
# anndata::write_h5ad(ann, "./data/McKellarMuscleData.h5ad")

In [ ]:

#| label: load-libs
#| message: false
#| warning: false
library(ggplot2)
library(RColorBrewer)
library(ExperimentHub)
library(SpatialExperiment)
library(STexampleData)

In [ ]:
#| label: load-data
#| message: false
eh <- ExperimentHub()
q <- query(eh, "MERFISH")
df <- eh[["EH7546"]]

In [ ]:
#| label: make-spe
# extract cell metadata
i <- seq_len(9)
cd <- data.frame(df[, i], row.names = 1)

# set sample identifiers
id <- grep("Bregma", names(cd))
names(cd)[id] <- "sample_id"

# rename spatial coordinates
xy <- grep("Centroid", names(cd))
xy <- names(cd)[xy] <- c("x", "y")

# simplify annotations
cd$cluster_id <- cd$Cell_class
for (. in c("Endothelial", "OD Mature", "OD Immature"))
  cd$cluster_id[grep(., cd$cluster_id)] <- .

# extract & sparsify assay data
y <- data.frame(df[, -i], row.names = df[, 1])
y <- as(t(as.matrix(y)), "dgCMatrix")

# construct SPE
(spe <- SpatialExperiment(
  assays = list(exprs  = y),
  spatialCoordsNames = xy,
  colData = cd))

In [ ]:

#| label: plot-data
gg <- data.frame(spatialCoords(spe), colData(spe))
pal <- brewer.pal(length(unique(gg$cluster_id)), "Paired")
ggplot(gg, aes(x, y, col = cluster_id)) +
  facet_wrap(~ sample_id, scales = "free") +
  geom_point(size = 0.1) + scale_color_manual(values = pal) +
  guides(col = guide_legend(override.aes = list(size = 2))) +
  theme_void() + theme(legend.key.size = unit(0.5, "lines"))

In [ ]:
#| label: save-data
# Define the directory and file paths
dir_path <- "./data"

# Check if the directory exists, and create it if it doesn't
if (!dir.exists(dir_path)) {
  dir.create(dir_path)
}

saveRDS(spe, "./data/spe.rds")